[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/basic_gp.ipynb)

# Gaussian process regression with `gpc`

GPc port of [`GPy/basic_gp.ipynb`](../GPy/basic_gp.ipynb). Modelling runs in the C++ core; this notebook is a thin NumPy / matplotlib boundary.

**Differences from GPy to watch for**

* `m.objective()` is **−log-likelihood** (SCG minimises it).
* Observation noise is `likelihood.variance` (CIP-0027); Bias/White on the kernel are opt-in.
* Parameters use `set_by_path` / `get_by_path` / `param_paths`, not `m["rbf.lengthscale"]`.
* 2-D Matérn ARD + GPy-style slicing plots are **deferred** until the façade exposes those kernels / plot helpers.

In [ ]:
try:
    import gpc
except ImportError:
    import subprocess
    import sys
    # Prefer PyPI; fall back to TestPyPI for pre-release alphas (GPc CIP-002C).
    cmd = [sys.executable, "-m", "pip", "install", "-q", "gpc"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--index-url",
                "https://test.pypi.org/simple/",
                "--extra-index-url",
                "https://pypi.org/simple/",
                "gpc>=0.1.0a6",
            ]
        )
    import gpc


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from gpc.plot import plot_1d


## 1-dimensional model

Toy data: noisy observations of a sine.

In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(-3.0, 3.0, size=(20, 1))
Y = np.sin(X) + rng.normal(scale=0.05, size=X.shape)

Build an RBF kernel and an FTC Gaussian regression model:

In [ ]:
kernel = gpc.kern.RBF(1, variance=1.0, lengthscale=1.0)
m = gpc.models.GPRegression(X, Y, kernel=kernel)
print(m)
print("paths:", m.param_paths())

Predictive mean ± 2σ before optimisation:

In [ ]:
plot_1d(m, X, Y, label="before optimize")
plt.show()

Optimise hyperparameters (C++ SCG). `messages=True` refreshes a paramz-style table in Jupyter:

In [ ]:
m.optimize(iters=100, messages=True)
print(m)
print("objective (−log lik):", m.objective())
plot_1d(m, X, Y, label="after optimize")
plt.show()

### Path-based parameter access

GPy: `m["rbf.lengthscale"] = 0.5`. Here:

In [ ]:
print("lengthscale:", m.get_by_path("rbf.lengthscale"))
m.set_by_path("rbf.lengthscale", 0.5)
print("after set:", m.get_by_path("rbf.lengthscale"))
m.optimize(iters=50)
print(m)

## 2-dimensional example (deferred)

The GPy twin uses `Matern52(2, ARD=True) + White(2)` and GPy's 2-D / slice plotting. The current `gpc.kern` surface is isotropic RBF / ExpQuad / MLP / ArcCosine (+ Bias/White). Revisit when ARD Matérn lands in the façade; until then use the [native GPc tutorials](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/README.md) or GPy for that cell.